**Data fields**\
Most of the fields are self-explanatory. The following are descriptions for those that aren't.

* `Id` - an Id that represents a (Store, Date) duple within the test set

* `Store` - a unique Id for each store

* `Sales` - the turnover for any given day (this is what you are predicting)

* `Customers` - the number of customers on a given day

* `Open` - an indicator for whether the store was open: 0 = closed, 1 = open

* `StateHoliday` - indicates a state holiday. Normally all stores, with few exceptions, are closed on state holidays. Note that all schools are closed on public holidays and weekends. a = public holiday, b = Easter holiday, c = Christmas, 0 = None

* `SchoolHoliday` - indicates if the (Store, Date) was affected by the closure of public schools

* `StoreType` - differentiates between 4 different store models: a, b, c, d

* `Assortment` - describes an assortment level: a = basic, b = extra, c = extended

* `CompetitionDistance` - distance in meters to the nearest competitor store

* `CompetitionOpenSince[Month/Year]` - gives the approximate year and month of the time the nearest competitor was opened

* `Promo` - indicates whether a store is running a promo on that day

* `Promo2` - Promo2 is a continuing and consecutive promotion for some stores: 0 = store is not participating, 1 = store is participating

* `Promo2Since[Year/Week]` - describes the year and calendar week when the store started participating in Promo2

* `PromoInterval` - describes the consecutive intervals Promo2 is started, naming the months the promotion is started anew. E.g. "Feb,May,Aug,Nov" means each round starts in February, May, August, November of any given year for that store


## Import Libraries

In [156]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# show all columns
pd.set_option('display.max_columns', None)

## Load datasets

In [157]:
train_df = pd.read_csv(r'./data/train.csv', low_memory=False)
test_df = pd.read_csv(r'./data/test.csv', low_memory=False)
store_df = pd.read_csv(r'./data/store.csv', low_memory=False)

## Cleaning and EDA

Let's take a look at all the DataFrames:

In [158]:
train_df

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1
...,...,...,...,...,...,...,...,...,...
1017204,1111,2,2013-01-01,0,0,0,0,a,1
1017205,1112,2,2013-01-01,0,0,0,0,a,1
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1


In [159]:
store_df

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1110,1111,a,a,1900.0,6.0,2014.0,1,31.0,2013.0,"Jan,Apr,Jul,Oct"
1111,1112,c,c,1880.0,4.0,2006.0,0,NaN,NaN,NaN
1112,1113,a,c,9260.0,NaN,NaN,0,NaN,NaN,NaN
1113,1114,a,c,870.0,NaN,NaN,0,NaN,NaN,NaN


**NOTES:**
- `Promo2` is store-level flag for a continuing, recurring promotion that some stores take part in, It describes the store, not the day.
- `Promo2Since[Year/Week]` describes the year and calendar week when the store started participating in Promo2
- `PromoInterval`: the months in which Promo2 starts a new round. For example, "Feb,May,Aug,Nov" means a new round begins in those months every year.

In [160]:
test_df

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0
...,...,...,...,...,...,...,...,...
41083,41084,1111,6,2015-08-01,1.0,0,0,0
41084,41085,1112,6,2015-08-01,1.0,0,0,0
41085,41086,1113,6,2015-08-01,1.0,0,0,0
41086,41087,1114,6,2015-08-01,1.0,0,0,0


Let's investigate the NULLs values:

In [163]:
train_df.isnull().sum()

Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

No Null values in `train_df`.

What about `store_df`?

In [164]:
store_df.isnull().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

Let's handle these one-by-none:

In [165]:
# CompetitionDistance missing values
store_df[pd.isnull(store_df['CompetitionDistance'])]

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
290,291,d,a,NaN,NaN,NaN,0,NaN,NaN,NaN
621,622,a,c,NaN,NaN,NaN,0,NaN,NaN,NaN
878,879,d,a,NaN,NaN,NaN,1,5.0,2013.0,"Feb,May,Aug,Nov"


Apparently this information is simply missing from the data. No particular pattern observed. In this case, Let's replace NaN with the max value\
(to indicate that competition is very far way):

In [166]:
# replace missing CompetitionDistance
store_df['CompetitionDistance'] = store_df['CompetitionDistance'].fillna(store_df['CompetitionDistance'].max())

Since, `Promo2SinceWeek` = `NULL` means that store never participated in Promos and same goes for `Promo2SinceYear`, `PromoInterval`;\
Let's replace these NULL values with 0s:

In [167]:
# replace missing Promo2SinceWeek, Promo2SinceYear, PromoInterval
store_df = store_df.fillna({'Promo2SinceWeek': 0, 'Promo2SinceYear': 0, 'PromoInterval':0}, inplace=True)

Let's investigate missing `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` values:

In [168]:
store_df[pd.isnull(store_df['CompetitionOpenSinceMonth'])]

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
11,12,a,c,1070.0,NaN,NaN,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
12,13,d,a,310.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov"
15,16,a,c,3270.0,NaN,NaN,0,0.0,0.0,0
18,19,a,c,3240.0,NaN,NaN,1,22.0,2011.0,"Mar,Jun,Sept,Dec"
21,22,a,a,1040.0,NaN,NaN,1,22.0,2012.0,"Jan,Apr,Jul,Oct"
...,...,...,...,...,...,...,...,...,...,...
1095,1096,a,c,1130.0,NaN,NaN,1,10.0,2014.0,"Mar,Jun,Sept,Dec"
1099,1100,a,a,540.0,NaN,NaN,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
1112,1113,a,c,9260.0,NaN,NaN,0,0.0,0.0,0
1113,1114,a,c,870.0,NaN,NaN,0,0.0,0.0,0


Missing values in `CompetitionOpenSince[Month/Year]` is apparently just missing information from the data or Rossmann Store didn't know genuinely. In this case, replacing these missing values with 0s would make complete sense:

In [169]:
store_df = store_df.fillna({'CompetitionOpenSinceMonth': 0, 'CompetitionOpenSinceYear': 0}, inplace=True)

Let's merge `train_df` and `store_df`(additional information about the stores) for further operations:

In [170]:
merged_df = pd.merge(train_df, store_df, how='left', on='Store')

Let's check the `Dtype` and `Non-Null Count` of `merged_df`:

In [171]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1017209 non-null  int64  
 1   DayOfWeek                  1017209 non-null  int64  
 2   Date                       1017209 non-null  str    
 3   Sales                      1017209 non-null  int64  
 4   Customers                  1017209 non-null  int64  
 5   Open                       1017209 non-null  int64  
 6   Promo                      1017209 non-null  int64  
 7   StateHoliday               1017209 non-null  str    
 8   SchoolHoliday              1017209 non-null  int64  
 9   StoreType                  1017209 non-null  str    
 10  Assortment                 1017209 non-null  str    
 11  CompetitionDistance        1017209 non-null  float64
 12  CompetitionOpenSinceMonth  1017209 non-null  float64
 13  CompetitionOpenSinceYea

Let's convert `Date` to its correct Dtype, we need it for `feature engineering`:

In [172]:
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

---

### Feature Engineering

Let's extract `Year`, `Month`, `Day` from `Date` column:

In [173]:
merged_df['Year'] = merged_df['Date'].dt.year
merged_df['Month'] = merged_df['Date'].dt.month
merged_df['Day'] = merged_df['Date'].dt.day

Since, we have extracted all necessary part from `Date` let drop it:

In [174]:
merged_df = merged_df.drop('Date', axis=1)

Let's compute the number of months for which a competitor has been open near the store:

In [175]:
merged_df['CompetitionOpen'] = 12*(merged_df['Year'] - merged_df['CompetitionOpenSinceYear']) + (merged_df['Month'] - merged_df['CompetitionOpenSinceMonth'])

Let's check opened stores with zero sales stores:

In [176]:
open_zero_sales = merged_df[(merged_df['Open'] == 1) & (merged_df['Sales'] == 0)]

In [177]:
open_zero_sales.head()

,Store,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,CompetitionOpen
86825,971,5,0,0,1,0,0,1,c,a,1140.0,5.0,2011.0,1,14.0,2012.0,"Mar,Jun,Sept,Dec",2015,5,15,48.0
142278,674,4,0,0,1,0,0,0,a,a,2640.0,12.0,2005.0,1,31.0,2013.0,"Feb,May,Aug,Nov",2015,3,26,111.0
196938,699,4,0,0,1,1,0,0,a,a,180.0,0.0,0.0,1,5.0,2013.0,"Jan,Apr,Jul,Oct",2015,2,5,24182.0
322053,708,3,0,0,1,1,0,0,c,c,11470.0,10.0,2009.0,1,18.0,2014.0,"Jan,Apr,Jul,Oct",2014,10,1,60.0
330176,357,1,0,0,1,0,0,0,a,a,2060.0,10.0,2008.0,0,0.0,0.0,0,2014,9,22,71.0


In [178]:
open_zero_sales.shape

(54, 21)

There're only 54 days in the data where there are opened store with no sales on working days, so we can assume that there were\
external factors involved like public protests, strikes, or demonstrations.

Let's check closed stores with zero sales stores as well:

In [179]:
merged_df[(merged_df['Open'] == 0) & (merged_df['Sales'] == 0)]

,Store,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,CompetitionOpen
291,292,5,0,0,0,1,0,1,a,a,1100.0,6.0,2009.0,0,0.0,0.0,0,2015,7,31,73.0
875,876,5,0,0,0,1,0,1,a,a,21790.0,4.0,2005.0,1,18.0,2015.0,"Feb,May,Aug,Nov",2015,7,31,123.0
1406,292,4,0,0,0,1,0,1,a,a,1100.0,6.0,2009.0,0,0.0,0.0,0,2015,7,30,73.0
1990,876,4,0,0,0,1,0,1,a,a,21790.0,4.0,2005.0,1,18.0,2015.0,"Feb,May,Aug,Nov",2015,7,30,123.0
2521,292,3,0,0,0,1,0,1,a,a,1100.0,6.0,2009.0,0,0.0,0.0,0,2015,7,29,73.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1017204,1111,2,0,0,0,0,a,1,a,a,1900.0,6.0,2014.0,1,31.0,2013.0,"Jan,Apr,Jul,Oct",2013,1,1,-17.0
1017205,1112,2,0,0,0,0,a,1,c,c,1880.0,4.0,2006.0,0,0.0,0.0,0,2013,1,1,81.0
1017206,1113,2,0,0,0,0,a,1,a,c,9260.0,0.0,0.0,0,0.0,0.0,0,2013,1,1,24157.0
1017207,1114,2,0,0,0,0,a,1,a,c,870.0,0.0,0.0,0,0.0,0.0,0,2013,1,1,24157.0


There're 172817 closed stores in the data. It is about 10% of the total amount of observations. To avoid any biased forecasts, Let's drop these values:

In [180]:
merged_df = merged_df[(merged_df['Open'] != 0) & (merged_df['Sales'] != 0)]

---

Let's check the stats:

In [182]:
merged_df.describe()

,Store,DayOfWeek,Sales,Customers,Open,Promo,SchoolHoliday,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,Year,Month,Day,CompetitionOpen
count,844338.000000,844338.000000,844338.000000,844338.000000,844338.0,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000
mean,558.421374,3.520350,6955.959134,762.777166,1.0,0.446356,0.193578,5640.427743,4.926482,1369.692738,0.498670,11.596159,1003.201259,2013.831945,5.845774,15.835706,7730.589779
std,321.730861,1.723712,3103.815515,401.194153,0.0,0.497114,0.395102,8580.824305,4.283634,935.556484,0.499999,15.308101,1005.874685,0.777271,3.323959,8.683392,11230.049970
min,1.000000,1.000000,46.000000,8.000000,1.0,0.000000,0.000000,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2013.000000,1.000000,1.000000,-31.000000
25%,280.000000,2.000000,4859.000000,519.000000,1.0,0.000000,0.000000,710.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2013.000000,3.000000,8.000000,29.000000
50%,558.000000,3.000000,6369.000000,676.000000,1.0,0.000000,0.000000,2330.000000,4.000000,2006.000000,0.000000,0.000000,0.000000,2014.000000,6.000000,16.000000,91.000000
75%,837.000000,5.000000,8360.000000,893.000000,1.0,1.000000,0.000000,6910.000000,9.000000,2011.000000,1.000000,22.000000,2012.000000,2014.000000,8.000000,23.000000,24163.000000
max,1115.000000,7.000000,41551.000000,7388.000000,1.0,1.000000,1.000000,75860.000000,12.000000,2015.000000,1.000000,50.000000,2015.000000,2015.000000,12.000000,31.000000,24187.000000
